# Ngôn Sắc Việt — Train Whisper (PhoWhisper-small) Audio Ba Na → Việt (654h Corpus)

Notebook này huấn luyện mô hình **speech_translation** (`vinai/PhoWhisper-small`, nghe tiếng Ba Na → xuất câu tiếng Việt) trực tiếp từ bộ **`cuong06/Bahnar_Vietnamese`** (`cc-by-nc-4.0`, 113.830 mẫu audio đã căn hàng với tiếng Việt) trên GPU Kaggle/Colab.

- Tự phát hiện GPU CUDA, giải mã audio qua PyAV/torchcodec/datasets.
- Chấm điểm `test_wer` và `test_cer` trên split `test`, ghi đầy đủ `ngonsacviet.json` với `trained_on_unverified: false` (nhờ nguồn có license `cc-by-nc-4.0` rõ ràng).
- Đóng gói `/kaggle/working/whisper-bdq-ready.zip` sẵn sàng nạp vào `ML_ASR_MODEL=models/whisper-bdq`.

In [ ]:
REPO_URL = "https://github.com/minhduy6868/vietlangai-model.git"
BRANCH = "main"
BASE_MODEL = "vinai/PhoWhisper-small"
MAX_TRAIN_CLIPS = 12000
MAX_TEST_CLIPS = 300
EPOCHS = 3
LR = 1e-5
BATCH = 16
GRAD_ACCUM = 2


In [ ]:
import shutil
import subprocess
import sys

import torch

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.check_output([smi], text=True))
if not torch.cuda.is_available():
    raise RuntimeError("Cần GPU CUDA (T4/P100) để train 654h Bahnar speech translation.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "datasets>=2.19,<4", "soundfile", "librosa", "av>=12",
    "transformers>=4.56,<5", "accelerate>=1.0", "evaluate", "jiwer",
])


In [ ]:
import json
import shutil
import subprocess
import time
from pathlib import Path

import torch
from datasets import Audio, load_dataset
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, Seq2SeqTrainer, Seq2SeqTrainingArguments

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "vietlangai-model"
if not (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])

OUT_DIR = WORK / "whisper-bdq"
ds = load_dataset("cuong06/Bahnar_Vietnamese", streaming=True)

def take_usable(split_name, limit):
    rows = []
    for row in ds[split_name].cast_column("audio", Audio(sampling_rate=16000)):
        dur = float(row.get("duration") or 0.0)
        txt = (row.get("text_vi") or "").strip()
        if 0.5 <= dur <= 28.0 and txt and row.get("audio") is not None:
            rows.append({"array": row["audio"]["array"], "text": txt})
            if len(rows) >= limit:
                break
    return rows

train_rows = take_usable("train", MAX_TRAIN_CLIPS)
val_rows = take_usable("validation", 300)
test_rows = take_usable("test", MAX_TEST_CLIPS)
print({"train": len(train_rows), "val": len(val_rows), "test": len(test_rows)}, flush=True)

processor = AutoProcessor.from_pretrained(BASE_MODEL)
used_base_model = BASE_MODEL
try:
    model = AutoModelForSpeechSeq2Seq.from_pretrained(BASE_MODEL).to("cuda")
except ValueError as exc:
    if BASE_MODEL == "vinai/PhoWhisper-small" and ("torch" in str(exc) or "v2.6" in str(exc)):
        print("PhoWhisper .bin bị chặn trên torch < 2.6, chuyển sang openai/whisper-small (safetensors):", exc, flush=True)
        used_base_model = "openai/whisper-small"
        processor = AutoProcessor.from_pretrained(used_base_model)
        model = AutoModelForSpeechSeq2Seq.from_pretrained(used_base_model).to("cuda")
    else:
        raise
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
processor.tokenizer.set_prefix_tokens(language="vi", task="transcribe")
start_id = model.config.decoder_start_token_id

def collate(batch):
    feats = processor.feature_extractor([b["array"] for b in batch], sampling_rate=16000, return_tensors="pt").input_features
    labels = processor.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
    ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
    if (ids[:, 0] == start_id).all():
        ids = ids[:, 1:]
    return {"input_features": feats, "labels": ids}

per_dev = max(1, BATCH // GRAD_ACCUM)
trainer = Seq2SeqTrainer(
    model=model,
    args=Seq2SeqTrainingArguments(
        output_dir=str(OUT_DIR / "checkpoints"),
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_ratio=0.1,
        per_device_train_batch_size=per_dev,
        per_device_eval_batch_size=per_dev,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        prediction_loss_only=True,
        remove_unused_columns=False,
        logging_steps=50,
        report_to=[],
    ),
    train_dataset=train_rows,
    eval_dataset=val_rows,
    data_collator=collate,
)
started = time.time()
trainer.train()

import sys
sys.path.insert(0, str(REPO_DIR))
from training.speech_metrics import error_rates
model.eval()
hyps = []
for i in range(0, len(test_rows), 16):
    feats = processor.feature_extractor([b["array"] for b in test_rows[i : i + 16]], sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
    with torch.inference_mode():
        ids = model.generate(feats, language="vi", task="transcribe", max_new_tokens=225)
    hyps.extend(processor.batch_decode(ids, skip_special_tokens=True))
rates = error_rates(hyps, [b["text"] for b in test_rows])

OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT_DIR, safe_serialization=True)
processor.save_pretrained(OUT_DIR)
meta = {
    "task": "speech_translation",
    "language": "bdq",
    "text_language": "vi",
    "whisper_language": "vi",
    "base_model": used_base_model.split("/")[-1],
    "train_clips": len(train_rows),
    "test_clips": len(test_rows),
    "test_wer": rates["wer"],
    "test_cer": rates["cer"],
    "trained_on_unverified": False,
    "minutes": round((time.time() - started) / 60, 1),
}
(OUT_DIR / "ngonsacviet.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(meta, flush=True)

ckpt_dir = OUT_DIR / "checkpoints"
if ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)
zip_file = shutil.make_archive(str(WORK / "whisper-bdq-ready"), "zip", root_dir=str(WORK), base_dir="whisper-bdq")
print("Đã đóng gói:", zip_file, flush=True)
